# 01 - Data preparation and genetic-algorithm search

Santos SSS mine detection, PROJ518 MSc dissertation.

This notebook performs the two preparation stages and the hyperparameter search:

1. offline geometric augmentation of the 1,170 originals into 4,680 files, one group per original;
2. a group-aware stratified split of the groups into a locked hold-out and a training pool;
3. the genetic algorithm over seven optimiser and loss hyperparameters.

Runtime: Colab with a T4 GPU. The dataset is expected as `santos_sss.zip` in Google Drive.

## Session setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

if not os.path.exists('/content/santos_sss'):
    os.system('unzip -q /content/drive/MyDrive/santos_sss.zip -d /content/santos_sss')

os.makedirs('/content/working', exist_ok=True)
os.makedirs('/kaggle', exist_ok=True)
if not os.path.exists('/kaggle/working'):
    os.symlink('/content/working', '/kaggle/working')

import glob
n = len(glob.glob('/content/santos_sss/**/*.jpg', recursive=True))
print(f"Dataset images found: {n}  (πρέπει: 1170)")
print("Symlink OK:", os.path.realpath('/kaggle/working'))

## Environment

In [ ]:
import os, glob, shutil, random, json
import numpy as np, pandas as pd

if not os.path.exists('/kaggle/working/yolov5'):
    os.system('git clone https://github.com/ultralytics/yolov5.git /kaggle/working/yolov5')
    os.system('pip install -r /kaggle/working/yolov5/requirements.txt')
    os.system('pip install albumentations')

os.chdir('/kaggle/working/yolov5')

BASE = '/content/santos_sss'
if not os.path.exists(BASE):
    raise FileNotFoundError("Δεν βρέθηκε το dataset - τρέξε πρώτα το Cell 0 (Colab setup)")
print("Setup OK. Dataset base:", BASE)

## Offline geometric augmentation

One original plus three augmented children per image, written as real files.
Boxes are transformed with the image. No photometric transform is applied, so the
acoustic signature is preserved. Each original and its children share a group id.

In [ ]:
import cv2
import albumentations as A
random.seed(42); np.random.seed(42)

SRC   = BASE
AUG   = '/kaggle/working/santos_augmented'
N_AUG = 3

if os.path.exists(AUG): shutil.rmtree(AUG)
os.makedirs(f'{AUG}/images', exist_ok=True)
os.makedirs(f'{AUG}/labels', exist_ok=True)

def label_for(img_path):
    lbl = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if not os.path.exists(lbl):
        lbl = img_path.rsplit('.', 1)[0] + '.txt'
    return lbl if os.path.exists(lbl) else None

def read_yolo(lbl_path):
    boxes, classes = [], []
    if lbl_path:
        for line in open(lbl_path):
            p = line.split()
            if len(p) == 5:
                x, y, w, h = (float(v) for v in p[1:])
                x = min(max(x, 0.0), 1.0)
                y = min(max(y, 0.0), 1.0)
                w = min(w, 2*x, 2*(1-x))
                h = min(h, 2*y, 2*(1-y))
                if w <= 0 or h <= 0:
                    continue
                classes.append(int(p[0]))
                boxes.append([x, y, w, h])
    return boxes, classes

def write_yolo(lbl_path, boxes, classes):
    with open(lbl_path, 'w') as f:
        for c, b in zip(classes, boxes):
            f.write(f"{c} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}\n")

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Affine(rotate=(-10, 10), translate_percent=(-0.10, 0.10),
             scale=(0.8, 1.2), border_mode=cv2.BORDER_CONSTANT, p=0.9),
], bbox_params=A.BboxParams(format='yolo', label_fields=['classes'],
                            min_visibility=0.3))

imgs = sorted(glob.glob(os.path.join(SRC, '**', '*.jpg'), recursive=True))
print(f"Found {len(imgs)} originals. Target: {len(imgs)*(1+N_AUG)} total.")

groups = {}
def save_pair(img, boxes, classes, stem, group_stem):
    cv2.imwrite(f'{AUG}/images/{stem}.jpg', img)
    write_yolo(f'{AUG}/labels/{stem}.txt', boxes, classes)
    groups[stem] = group_stem

for img_path in imgs:
    stem = os.path.basename(img_path).rsplit('.', 1)[0]
    img  = cv2.imread(img_path)
    boxes, classes = read_yolo(label_for(img_path))
    save_pair(img, boxes, classes, stem, stem)
    made = tries = 0
    while made < N_AUG and tries < N_AUG * 5:
        tries += 1
        t = transform(image=img, bboxes=boxes, classes=classes)
        if boxes and len(t['bboxes']) == 0:
            continue
        save_pair(t['image'], [list(b) for b in t['bboxes']],
                  list(t['classes']), f'{stem}_aug{made}', stem)
        made += 1

json.dump(groups, open(f'{AUG}/groups.json', 'w'))
print(f"Done. Total files: {len(glob.glob(f'{AUG}/images/*.jpg'))}")
print(f"Group map saved -> {AUG}/groups.json")

## Group-aware stratified split

The draw is over the 1,170 original groups, stratified by annotation content, then each
group is expanded into its four files. A child can therefore never appear on the other
side of a split from its parent.

The fold count set in this cell is not used by the search below: the genetic algorithm
uses its own fixed 85/15 split inside the training pool. The 44-fold scheme reported in
the dissertation is built in notebook `02_cross_validation.ipynb`.

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
random.seed(42); np.random.seed(42)

AUG = '/kaggle/working/santos_augmented'
groups = json.load(open(f'{AUG}/groups.json'))
all_imgs = sorted(glob.glob(f'{AUG}/images/*.jpg'))

def content_label(img_path):
    lbl = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    hm = hn = False
    if os.path.exists(lbl):
        for line in open(lbl):
            line = line.strip()
            if not line: continue
            c = int(line.split()[0])
            if c == 0: hm = True
            elif c == 1: hn = True
    if hm and hn: return 3
    if hm:        return 1
    if hn:        return 2
    return 0

group_ids   = sorted(set(groups.values()))
group_label = {g: content_label(f'{AUG}/images/{g}.jpg') for g in group_ids}

g_arr = np.array(group_ids)
y_arr = np.array([group_label[g] for g in group_ids])
print("Groups:", len(g_arr), "| label counts:",
      {int(k): int((y_arr==k).sum()) for k in np.unique(y_arr)})

tv_groups, test_groups = train_test_split(
    g_arr, test_size=0.10, stratify=y_arr, random_state=42)
test_groups = set(test_groups); tv_groups = set(tv_groups)

def files_of(group_set):
    return np.array([p for p in all_imgs
                     if groups[os.path.basename(p).rsplit('.',1)[0]] in group_set])

test_imgs = files_of(test_groups)
tv_imgs   = files_of(tv_groups)
print(f"Hold-out test files: {len(test_imgs)} | train+val files: {len(tv_imgs)}")

tv_group_list = sorted(tv_groups)
tv_y = np.array([group_label[g] for g in tv_group_list])
skf = StratifiedKFold(n_splits=40, shuffle=True, random_state=42)

tv_group_of = np.array([groups[os.path.basename(p).rsplit('.',1)[0]] for p in tv_imgs])
folds = []
for tr_g_idx, va_g_idx in skf.split(tv_group_list, tv_y):
    tr_groups = set(np.array(tv_group_list)[tr_g_idx])
    va_groups = set(np.array(tv_group_list)[va_g_idx])
    tr_mask = np.array([g in tr_groups for g in tv_group_of])
    va_mask = np.array([g in va_groups for g in tv_group_of])
    folds.append((np.where(tr_mask)[0], np.where(va_mask)[0]))

print(f"Built {len(folds)} folds. Example fold sizes: "
      f"train={len(folds[0][0])}, val={len(folds[0][1])}")

## Hyperparameter files and helpers

The baseline configuration is the YOLOv5 defaults with on-the-fly augmentation switched
off, because augmentation was already done offline. The two configurations differ only
in the tuned values.

In [ ]:
EPOCHS_CV = 40
IMG       = 512
results_path = '/kaggle/working/cv_results.csv'

ga_hyp = 'lr0: 0.004371\nlrf: 0.083805\nmomentum: 0.937932\nweight_decay: 0.000639\nwarmup_epochs: 3.0\nwarmup_momentum: 0.8\nwarmup_bias_lr: 0.1\nbox: 0.045784\ncls: 0.240395\ncls_pw: 1.0\nobj: 1.948448\nobj_pw: 1.0\niou_t: 0.20\nanchor_t: 4.0\nfl_gamma: 0.0\nhsv_h: 0.0\nhsv_s: 0.0\nhsv_v: 0.0\ndegrees: 0.0\ntranslate: 0.0\nscale: 0.0\nshear: 0.0\nperspective: 0.0\nflipud: 0.0\nfliplr: 0.0\nmosaic: 0.0\nmixup: 0.0\ncopy_paste: 0.0\n'
with open('/kaggle/working/hyp_ga_best.yaml','w') as f: f.write(ga_hyp)

baseline_hyp = 'lr0: 0.01\nlrf: 0.01\nmomentum: 0.937\nweight_decay: 0.0005\nwarmup_epochs: 3.0\nwarmup_momentum: 0.8\nwarmup_bias_lr: 0.1\nbox: 0.05\ncls: 0.5\ncls_pw: 1.0\nobj: 1.0\nobj_pw: 1.0\niou_t: 0.20\nanchor_t: 4.0\nfl_gamma: 0.0\nhsv_h: 0.0\nhsv_s: 0.0\nhsv_v: 0.0\ndegrees: 0.0\ntranslate: 0.0\nscale: 0.0\nshear: 0.0\nperspective: 0.0\nflipud: 0.0\nfliplr: 0.0\nmosaic: 0.0\nmixup: 0.0\ncopy_paste: 0.0\n'
with open('/kaggle/working/hyp_baseline.yaml','w') as f: f.write(baseline_hyp)

GA_HYP   = '/kaggle/working/hyp_ga_best.yaml'
BASE_HYP = '/kaggle/working/hyp_baseline.yaml'

def empty_txt_for_backgrounds(img_list, labels_dir):
    for p in img_list:
        stem = os.path.basename(p).rsplit('.', 1)[0]
        src  = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        dst  = os.path.join(labels_dir, stem + '.txt')
        if os.path.exists(src): shutil.copy(src, dst)
        else: open(dst, 'w').close()

def make_split_dir(name, train_imgs, val_imgs):
    root = f'/kaggle/working/data_{name}'
    if os.path.exists(root): shutil.rmtree(root)
    for sub in ['images/train','images/val','labels/train','labels/val']:
        os.makedirs(os.path.join(root, sub), exist_ok=True)
    for p in train_imgs: shutil.copy(p, os.path.join(root,'images/train',os.path.basename(p)))
    for p in val_imgs:   shutil.copy(p, os.path.join(root,'images/val',os.path.basename(p)))
    empty_txt_for_backgrounds(train_imgs, os.path.join(root,'labels/train'))
    empty_txt_for_backgrounds(val_imgs,   os.path.join(root,'labels/val'))
    yp = os.path.join(root,'data.yaml')
    with open(yp,'w') as f:
        f.write(f"train: {root}/images/train\nval: {root}/images/val\nnc: 2\nnames: ['MILCO','NOMBO']\n")
    return yp

def parse_results(run_dir):
    df = pd.read_csv(os.path.join(run_dir,'results.csv'))
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    P = float(last['metrics/precision']); R = float(last['metrics/recall'])
    f1 = 2*P*R/(P+R) if (P+R) > 0 else 0.0
    return {'precision': P, 'recall': R, 'f1': f1,
            'mAP50': float(last['metrics/mAP_0.5']),
            'mAP50_95': float(last['metrics/mAP_0.5:0.95'])}

def already_done(config, fold):
    if not os.path.exists(results_path): return False
    d = pd.read_csv(results_path)
    return ((d['config']==config) & (d['fold']==fold)).any()

def save_row(m):
    rows = pd.read_csv(results_path).to_dict('records') if os.path.exists(results_path) else []
    rows.append(m); pd.DataFrame(rows).to_csv(results_path, index=False)

def run_one(config, fold, epochs=EPOCHS_CV):
    if already_done(config, fold):
        print(f"  {config} fold{fold}: already done - skip"); return
    tr, va = folds[fold]
    yp = make_split_dir(f'fold{fold}', tv_imgs[tr], tv_imgs[va])
    hyp = GA_HYP if config == 'ga' else BASE_HYP
    name = f'{config}_fold{fold}'
    log  = f'/kaggle/working/log_{name}.txt'
    cmd = (f'python train.py --img {IMG} --batch 16 --epochs {epochs} --data {yp} '
           f'--weights yolov5s.pt --hyp {hyp} --name {name} --project /kaggle/working/runs '
           f'--exist-ok --cache > {log} 2>&1')
    print(f"  running {name} ... -> {log}  (~30 min)")
    os.system(cmd)
    rc = f'/kaggle/working/runs/{name}/results.csv'
    if not os.path.exists(rc):
        print(f"  !! {name} FAILED. Last log lines:")
        print(''.join(open(log).readlines()[-12:])); return
    m = parse_results(f'/kaggle/working/runs/{name}'); m.update({'config':config,'fold':fold})
    save_row(m)
    print(f"  DONE {name}: mAP50={m['mAP50']:.3f} R={m['recall']:.3f} P={m['precision']:.3f} F1={m['f1']:.3f}")

def run_fold(fold):
    print(f"=== FOLD {fold} ===")
    run_one('baseline', fold)
    run_one('ga', fold)
    print(f"=== fold {fold} complete ===")

print("Runner ready (GA hyp = final values, mAP=0.674).")

## Genetic algorithm

Population 6, five generations, fitness is mAP@0.5 after 20 epochs on a fixed 85/15 split
inside the training pool. Tournament selection, single-point crossover, Gaussian mutation,
elitism, and a fitness cache so a repeated chromosome is not retrained. The locked hold-out
is never touched here. The best-so-far chromosome is written after every generation, so a
session cut-off does not lose the search.

In [ ]:
import itertools
random.seed(42); np.random.seed(42)

POP, GENS, GA_EPOCHS = 6, 5, 20
TOURNAMENT_K = 3
P_CROSS, P_MUT, MUT_SIGMA = 0.8, 0.2, 0.10

GENES = {
    'lr0':          (0.001,  0.01),
    'lrf':          (0.001,  0.1),
    'momentum':     (0.85,   0.98),
    'weight_decay': (0.0001, 0.001),
    'box':          (0.02,   0.2),
    'cls':          (0.1,    1.0),
    'obj':          (0.5,    2.0),
}
GENE_NAMES = list(GENES.keys())
LOWS  = np.array([GENES[g][0] for g in GENE_NAMES])
HIGHS = np.array([GENES[g][1] for g in GENE_NAMES])

ga_cache_path = '/kaggle/working/ga_cache.csv'

ga_tr_idx, ga_va_idx = train_test_split(
    np.arange(len(tv_imgs)), test_size=0.15, random_state=123)
GA_TRAIN = tv_imgs[ga_tr_idx]
GA_VAL   = tv_imgs[ga_va_idx]
print(f"GA split: train={len(GA_TRAIN)}  val={len(GA_VAL)}  (test_imgs untouched)")
GA_DATA = make_split_dir('ga_search', GA_TRAIN, GA_VAL)

def vec_to_hyp(vec, path):
    h = dict(zip(GENE_NAMES, [float(v) for v in vec]))
    lines = [
        f"lr0: {h['lr0']:.6f}", f"lrf: {h['lrf']:.6f}",
        f"momentum: {h['momentum']:.6f}", f"weight_decay: {h['weight_decay']:.6f}",
        "warmup_epochs: 3.0", "warmup_momentum: 0.8", "warmup_bias_lr: 0.1",
        f"box: {h['box']:.6f}", f"cls: {h['cls']:.6f}", "cls_pw: 1.0",
        f"obj: {h['obj']:.6f}", "obj_pw: 1.0",
        "iou_t: 0.20", "anchor_t: 4.0", "fl_gamma: 0.0",
        "hsv_h: 0.0", "hsv_s: 0.0", "hsv_v: 0.0",
        "degrees: 0.0", "translate: 0.0", "scale: 0.0",
        "shear: 0.0", "perspective: 0.0",
        "flipud: 0.0", "fliplr: 0.0", "mosaic: 0.0", "mixup: 0.0", "copy_paste: 0.0",
    ]
    with open(path, 'w') as f: f.write("\n".join(lines) + "\n")

def vec_key(vec): return "|".join(f"{v:.5f}" for v in vec)

def cache_lookup(vec):
    if not os.path.exists(ga_cache_path): return None
    d = pd.read_csv(ga_cache_path)
    hit = d[d['key'] == vec_key(vec)]
    return float(hit['fitness'].iloc[0]) if len(hit) else None

def cache_store(vec, fit, gen):
    row = {'key': vec_key(vec), 'fitness': fit, 'gen': gen}
    for n, v in zip(GENE_NAMES, vec): row[n] = float(v)
    rows = pd.read_csv(ga_cache_path).to_dict('records') if os.path.exists(ga_cache_path) else []
    rows.append(row); pd.DataFrame(rows).to_csv(ga_cache_path, index=False)

ga_eval_counter = itertools.count()
def fitness(vec, gen):
    cached = cache_lookup(vec)
    if cached is not None:
        print(f"    cache hit: mAP={cached:.3f}"); return cached
    i = next(ga_eval_counter)
    hyp = f'/kaggle/working/hyp_ga_eval_{i}.yaml'; vec_to_hyp(vec, hyp)
    name = f'ga_eval_{i}'; log = f'/kaggle/working/log_{name}.txt'
    cmd = (f'python train.py --img {IMG} --batch 16 --epochs {GA_EPOCHS} --data {GA_DATA} '
           f'--weights yolov5s.pt --hyp {hyp} --name {name} --project /kaggle/working/runs '
           f'--exist-ok --cache > {log} 2>&1')
    os.system(cmd)
    rc = f'/kaggle/working/runs/{name}/results.csv'
    if not os.path.exists(rc):
        print(f"    eval {i} FAILED -> fitness 0"); cache_store(vec, 0.0, gen); return 0.0
    fit = parse_results(f'/kaggle/working/runs/{name}')['mAP50']
    cache_store(vec, fit, gen)
    print(f"    eval {i}: mAP={fit:.3f}")
    return fit

def random_individual():
    return LOWS + np.random.rand(len(GENE_NAMES)) * (HIGHS - LOWS)

def tournament(pop, fits):
    idx = np.random.choice(len(pop), TOURNAMENT_K, replace=False)
    best = idx[np.argmax([fits[j] for j in idx])]
    return pop[best].copy()

def crossover(a, b):
    if np.random.rand() > P_CROSS: return a.copy(), b.copy()
    pt = np.random.randint(1, len(GENE_NAMES))
    return (np.concatenate([a[:pt], b[pt:]]),
            np.concatenate([b[:pt], a[pt:]]))

def mutate(vec):
    out = vec.copy()
    for j in range(len(out)):
        if np.random.rand() < P_MUT:
            sigma = MUT_SIGMA * (HIGHS[j] - LOWS[j])
            out[j] = np.clip(out[j] + np.random.randn() * sigma, LOWS[j], HIGHS[j])
    return out

def run_ga():
    pop = [random_individual() for _ in range(POP)]
    history = []; best_vec, best_fit = None, -1.0
    for gen in range(GENS):
        print(f"\n=== GA GENERATION {gen+1}/{GENS} ===")
        fits = [fitness(ind, gen) for ind in pop]
        gbest = int(np.argmax(fits))
        if fits[gbest] > best_fit:
            best_fit, best_vec = fits[gbest], pop[gbest].copy()
        history.append({'gen': gen+1, 'best': max(fits), 'avg': float(np.mean(fits))})
        print(f"  gen {gen+1}: best mAP={max(fits):.3f}  avg mAP={np.mean(fits):.3f}")
        vec_to_hyp(best_vec, '/kaggle/working/hyp_ga_best.yaml')
        pd.DataFrame(history).to_csv('/kaggle/working/ga_convergence.csv', index=False)
        print(f"  [saved] best-so-far hyp (best mAP={best_fit:.3f})")
        new_pop = [best_vec.copy()]
        while len(new_pop) < POP:
            p1, p2 = tournament(pop, fits), tournament(pop, fits)
            c1, c2 = crossover(p1, p2)
            new_pop.append(mutate(c1))
            if len(new_pop) < POP: new_pop.append(mutate(c2))
        pop = new_pop
    return best_vec, best_fit, pd.DataFrame(history)

best_vec, best_fit, ga_history = run_ga()
print("\n==================== GA DONE ====================")
print(f"Best val mAP@0.5: {best_fit:.3f}")
for n, v in zip(GENE_NAMES, best_vec): print(f"  {n}: {v:.6f}")
vec_to_hyp(best_vec, '/kaggle/working/hyp_ga_best.yaml')
ga_history.to_csv('/kaggle/working/ga_convergence.csv', index=False)
print("\nSaved -> hyp_ga_best.yaml  and  ga_convergence.csv")